# ClickHouse API Tutorial — Session Purchase Forecasting (Oct 2019)

## Objective (Learn ClickHouse in ~60 minutes)
We build a small analytics platform using **ClickHouse** as the core OLAP engine.

- **Dataset**: `data/2019-Oct.csv` (event log)
- **Hypothesis**: Using the **first N events** in a `user_session` (start with **N=5**), we can predict whether the session will have a **purchase later**.
- **Label**: `has_purchase_in_session` $\in$ {0,1}

## What this notebook does
- Creates a robust raw -> typed ingestion pipeline in ClickHouse (handles `event_time` strings like `... UTC`)
- Runs OLAP/funnel analytics queries
- Builds a physical ML training table in ClickHouse: `training_sessions_n5`

## Step 1/10 — Setup + connect

**What this does**: Imports libraries, points Python at `/app` (so we can import helper code), and opens a ClickHouse connection.

**Why we do it**: Every step below is driven by SQL. If we can connect and run a trivial query, we know the environment is healthy.

**What to look for in the output (and how to interpret it)**:
- **ClickHouse version**: confirms you’re connected to the server you think you are.
- **Database**: should be `ecomm`.

**Common issues + fixes**:
- **Connection refused / timeout**: start the container: `docker compose up -d clickhouse`.
- **Wrong database**: check `CLICKHOUSE_DB` env var and defaults in `clickhouse_utils.py`.

In [16]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path("/app")))


import pandas as pd
import plotly.express as px

from clickhouse_utils import get_client, print_kv, ch_df

print("Step 1/10: Setup + connect")

DATA_PATH = Path("/app/data/2019-Oct.csv")
print_kv("Dataset path", DATA_PATH)

client = get_client()
print_kv("ClickHouse version", client.execute("SELECT version()"))
print_kv("Database", client.execute("SELECT currentDatabase()"))

Step 1/10: Setup + connect
Dataset path: /app/data/2019-Oct.csv
ClickHouse version: [('26.3.9.8',)]
Database: [('ecomm',)]


## Step 3/10 — Confirm ingestion happened (fast check)

**What this does**: Verifies that the dataset was ingested into ClickHouse by checking row counts in `events_raw` and `events_typed`.

**Why we do it**: The CSV is large; ingestion is intentionally done by `./docker_ingest.sh` (fast, reproducible) instead of loading gigabytes into pandas. This check prevents “mystery failures” later.

**What to look for in the output (and how to interpret it)**:
- **Both counts > 0**: ingestion actually happened.
- **Counts match**: the raw→typed materialized view is working and successfully parsed timestamps.

**Common issues + fixes**:
- **Table not found / zero rows**: run `./docker_ingest.sh` from project root, then rerun from Step 1.
- **Raw > typed**: some timestamps failed parsing. Re-check the MV logic (timestamp parsing + `UTC` string cleanup) in `docker_ingest.sh`.


In [17]:
print("Step 3/10: Verify data is already ingested (fast)")

# For a linear, reproducible run we ingest with ClickHouse directly:
#   ./docker_ingest.sh
# This keeps the notebook fast and avoids loading a multi-GB CSV in Python.

raw_rows = client.execute("SELECT count() FROM events_raw")[0][0]
typed_rows = client.execute("SELECT count() FROM events_typed")[0][0]
print_kv("events_raw rows", raw_rows)
print_kv("events_typed rows", typed_rows)

if raw_rows == 0 or typed_rows == 0:
    raise RuntimeError(
        "Data not ingested yet. Run: ./docker_ingest.sh (from project root), then rerun this notebook."
    )

print("Data is ready. Continuing with analytics + feature engineering...")


Step 3/10: Verify data is already ingested (fast)
events_raw rows: 42448764
events_typed rows: 42448764
Data is ready. Continuing with analytics + feature engineering...
